# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one pseudonymized content item (page) belonging to one client. `content_id` is unique per row (verified below); `client_id` groups pages under their 32 clients.

**Time window:** every `_90d` column is a trailing 90-day window ending at export time (no absolute dates in this slice — the window is expressed in days, not calendar dates). Two 30-day windows sit *inside* that 90-day window and are used for the trend calculation: `*_last_30d` (most recent 30 days) and `*_prev_30d` (the 30 days before that, i.e. days 31–60 back). These two 30-day windows do not cover the full 90 days — there are ~30 days of history (days 61–90 back) that feed the 90d totals but aren't in either comparison window.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df)}")
print(f"Unique content_id: {df.content_id.nunique()}")

# Grain check: content_id should never repeat
dupe_grain = df.groupby("content_id").size()
print(f"content_id values with >1 row: {(dupe_grain > 1).sum()}")

print(f"Distinct clients: {df.client_id.nunique()}")

# Window check: days_with_impressions/days_with_sessions should sit within 0-90
print(f"days_with_impressions range: {df.days_with_impressions.min()}-{df.days_with_impressions.max()}")
print(f"days_with_sessions range: {df.days_with_sessions.min()}-{df.days_with_sessions.max()}")

Rows: 30000
Unique content_id: 30000
content_id values with >1 row: 0
Distinct clients: 32
days_with_impressions range: 1-88
days_with_sessions range: 1-90


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

My lane is **CTR / Engagement Opportunity Scoring**: for pages that are visible in search, flag the ones underperforming their own position tier's median CTR — a candidate refresh queue.

**Label / proxy** (what defines "opportunity" — never a feature)
- `ctr` — the metric the opportunity flag is built from
- `clicks_90d` — the numerator inside `ctr`; using it as a feature would let the model see its own target

**Context** (for grouping/joining/filtering the analysis — never fed to a model)
- `content_id`, `client_id` — pseudonyms, joins and client-holdout splits only
- `avg_position`, `position_tier` — used here to define each page's peer group (the tier its CTR gets compared against), not as a learned input
- `impression_tier`, `impressions_90d` — used to build the eligible pool (visible, ≥500 impressions/90d)

**Feature** (knowable independent of the CTR outcome, safe to use)
- `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`
- `word_count`, `char_count`, `word_count_tier`, `char_count_tier`
- `content_age_days`, `age_tier`, `days_since_last_update`, `freshness_tier`
- `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `ai_sessions_90d`, `sessions_90d`, `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`

**Excluded** (each with a why)
- `trend_direction`, `trend_pct` — the pipeline's decline label is computed from these; keeping them out avoids quietly redefining "CTR opportunity" as "traffic is trending down," which is a different claim
- `provider_used`, `model_used` — data dictionary marks these "not a model feature"; which LLM generated the copy isn't something an editor can act on, and it risks encoding provider-quality bias rather than genuine content signal
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — these exist only to build `trend_pct`/`trend_direction`; excluded for the same reason those are

In [2]:
# Confirm the label/proxy relationship holds: ctr should reconstruct from clicks_90d / impressions_90d
check = df[df.impressions_90d > 0].copy()
check["ctr_recomputed"] = check.clicks_90d / check.impressions_90d * 100
diff = (check.ctr - check.ctr_recomputed).abs()
print(f"Max abs diff between stored ctr and clicks/impressions*100: {diff.max():.4f}")

# Confirm trend_direction / trend_pct really are downstream of the 30d windows, not independent signals
trend_check = df[df.impressions_prev_30d > 0].copy()
trend_check["trend_pct_recomputed"] = (
    (trend_check.impressions_last_30d - trend_check.impressions_prev_30d)
    / trend_check.impressions_prev_30d * 100
)
trend_diff = (trend_check.trend_pct - trend_check.trend_pct_recomputed).abs()
print(f"Max abs diff between stored trend_pct and recomputed: {trend_diff.max():.4f}")
print("-> confirms trend_pct/trend_direction are derived, not primary signal: excluded is correct.")

Max abs diff between stored ctr and clicks/impressions*100: 0.0050
Max abs diff between stored trend_pct and recomputed: 0.0500
-> confirms trend_pct/trend_direction are derived, not primary signal: excluded is correct.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain and window claims are verified in the cell under Section 1. The cell above verifies the label/proxy relationship. The cell below verifies missingness — confirming it follows `content_type`, not random, per the flyrank-data skill's warning.

In [3]:
keyword_cols = ["search_volume", "competition", "competition_level", "cpc"]
print("Missing keyword-context columns overall:")
print(df[keyword_cols].isna().sum())
print()

print("Missing search_volume by content_type (should be patterned, not uniform):")
print(df.groupby("content_type")["search_volume"].apply(lambda s: s.isna().mean()))
print()

print(f"Missing word_count/char_count overall: {df.word_count.isna().sum()} / {df.char_count.isna().sum()}")
print("Missing word_count by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: s.isna().mean()))
print()

print(f"avg_position == 0 ('no data', not rank zero): {(df.avg_position == 0).sum()} rows")
print(f"scroll_rate blank (pageviews_90d == 0): {df.scroll_rate.isna().sum()} rows, "
      f"matches pageviews_90d==0 count: {(df.pageviews_90d == 0).sum()}")

Missing keyword-context columns overall:
search_volume        2468
competition          2468
competition_level    2610
cpc                  2468
dtype: int64

Missing search_volume by content_type (should be patterned, not uniform):
content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.013673
Name: search_volume, dtype: float64

Missing word_count/char_count overall: 7699 / 7699
Missing word_count by content_type:
content_type
comparison article    0.000000
feedly article        0.000000
keyword article       0.282979
Name: word_count, dtype: float64

avg_position == 0 ('no data', not rank zero): 1205 rows
scroll_rate blank (pageviews_90d == 0): 125 rows, matches pageviews_90d==0 count: 125


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No calendar dates in this slice** — only day-counts (`_90d`, `_last_30d`, `_prev_30d`). Two pages both showing "90 days" of data are not necessarily the same 90 calendar days; this rules out any claim that ties the opportunity flag to a specific season or event.
- **`avg_position = 0` is missing data, not a genuinely ranked position** (1,205 rows) — excluded from the eligible pool rather than treated as position zero, or the peer-tier comparison collapses.
- **Keyword-context fields are missing entirely for one content type** (`feedly article`, 100% missing `search_volume`/`competition`/`cpc`) — the contract can flag CTR opportunity for those pages, but can never explain the opportunity in terms of keyword difficulty or search volume for that slice.
- **This is a 30k-row single-snapshot teaching slice**, not the full warehouse — it can't show whether a page's opportunity status is stable over time (that needs `fact_content_daily_performance` in the warehouse release, weeks 3+).
- **Association, not causation** — "below tier median CTR" is a descriptive gap, not evidence that a refresh *causes* CTR to rise; the data has no experiment/control to support that.

In [4]:
# feedly article: keyword fields are ALL missing, not just some -> can never feature-engineer keyword signal for this slice
feedly = df[df.content_type == "feedly article"]
print(f"feedly article rows: {len(feedly)}")
print(feedly[["search_volume", "competition", "cpc"]].isna().mean())
print()

# avg_position == 0 excluded from pool -> confirm pool definition already drops them
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
print(f"Eligible pool (visible, impressions_90d>=500, position<=20): {len(pool)} rows")
assert (pool.avg_position > 0).all(), "pool should never include avg_position==0 rows"

feedly article rows: 2096
search_volume    1.0
competition      1.0
cpc              1.0
dtype: float64

Eligible pool (visible, impressions_90d>=500, position<=20): 12023 rows


## 5. Output

*What does this analysis hand to the human, in one sentence?*

A ranked list of pages (`content_id`, `client_id`, `position_tier`, `ctr`, `tier_median_ctr`) that are visible in search, get meaningful volume, and sit below their own position tier's median CTR — for an editor to prioritize for a content refresh, not a system that acts on its own.

In [5]:
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
opportunity_queue = pool[pool.ctr < pool.tier_median_ctr].sort_values("ctr")

print(f"Eligible pool: {len(pool)} pages")
print(f"Opportunity queue (below tier median CTR): {len(opportunity_queue)} pages "
      f"({len(opportunity_queue)/len(pool):.1%} of the pool)")
opportunity_queue[["content_id", "client_id", "position_tier", "ctr", "tier_median_ctr"]].head()

Eligible pool: 12023 pages
Opportunity queue (below tier median CTR): 5888 pages (49.0% of the pool)


,content_id,client_id,position_tier,ctr,tier_median_ctr
8960,content_483e3d6fede3,client_7f2253d7e2,striking,0.0,0.17
22230,content_7c8916bd0e47,client_4e07408562,page_1,0.0,0.24
22184,content_d139b4205fbf,client_7f2253d7e2,striking,0.0,0.17
22183,content_0b061c56fc66,client_19581e27de,striking,0.0,0.17
22141,content_c4133119f4da,client_19581e27de,page_1,0.0,0.24


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.